# SAIFEN — 01 · Exploração inicial

Primeiro contato com a base **CelularesSubtraidos_2026.xlsx** da SSP-SP.

**Objetivos:**

1. Carregar a sheet de dados (`CELULAR_2026`) com o pacote `saifen_pipeline`.
2. Entender estrutura (linhas, colunas, tipos).
3. Mapear colunas-chave para o modelo do frontend (tipo de crime, período, geo).
4. Identificar problemas comuns (nulos, outliers de coordenadas, duplicidades).

> Rode `pip install -r ../requirements.txt` antes de abrir este notebook.

In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from saifen_pipeline import loader, config

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("Raw files disponíveis:")
for f in loader.list_raw_files():
    print(" -", f.name)

## 1. Carregar a base bruta

A SSP entrega a base em uma sheet `CELULAR_<ANO>`. `loader.load_ssp_xlsx`
detecta a sheet automaticamente.

In [ ]:
raw_path = config.RAW_DIR / "CelularesSubtraidos_2026.xlsx"
df_raw = loader.load_ssp_xlsx(raw_path)

print(f"Shape: {df_raw.shape}")
print(f"Source sheet: {df_raw.attrs.get('source_sheet')}")
df_raw.head(3)

## 2. Qualidade dos dados geográficos

A SSP avisa: a base é bruta, com possíveis duplicidades. Antes do KDE
precisamos saber **quantas linhas têm lat/lng utilizáveis**.

In [ ]:
lat = pd.to_numeric(df_raw[config.SSP_LAT_COL], errors="coerce")
lng = pd.to_numeric(df_raw[config.SSP_LNG_COL], errors="coerce")

total = len(df_raw)
n_null_geo = (lat.isna() | lng.isna()).sum()
n_zero = ((lat == 0) | (lng == 0)).sum()

min_lng, min_lat, max_lng, max_lat = config.SP_BBOX
inside = lat.between(min_lat, max_lat) & lng.between(min_lng, max_lng)
n_inside = inside.sum()

print(f"Total ......................... {total:>7,}")
print(f"Sem lat/lng ................... {n_null_geo:>7,} ({n_null_geo/total:.1%})")
print(f"Lat ou Lng == 0 ............... {n_zero:>7,}")
print(f"Dentro da bbox SP capital ..... {n_inside:>7,} ({n_inside/total:.1%})")

## 3. Categorias de crime e período

`RUBRICA` precisa ser mapeada para `furto` / `roubo` (vocabulário do frontend).
`HORA_OCORRENCIA` define o período (manhã / tarde / noite / madrugada).

In [ ]:
rubrica_counts = df_raw[config.SSP_RUBRICA_COL].value_counts().head(15)
display(rubrica_counts.to_frame("ocorrências"))

if "DESCR_PERIODO" in df_raw.columns:
    print("\nDESCR_PERIODO:")
    print(df_raw["DESCR_PERIODO"].value_counts(dropna=False).head(10))

## 4. Distribuição espacial bruta (scatter)

Plot rápido — só pra confirmar que os pontos válidos formam o desenho
de São Paulo antes de aplicarmos KDE.

In [ ]:
geo_ok = df_raw.assign(lat=lat, lng=lng).loc[inside, ["lat", "lng"]]

sample = geo_ok.sample(min(20_000, len(geo_ok)), random_state=42)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(sample["lng"], sample["lat"], s=1, alpha=0.15, c="#39FF14")
ax.set_facecolor("#000000")
fig.patch.set_facecolor("#000000")
ax.set_title("Subtração de celulares — SP capital (amostra 20k)", color="#00FF9F")
ax.tick_params(colors="#00FF9F")
ax.set_xlabel("Longitude", color="#00FF9F")
ax.set_ylabel("Latitude", color="#00FF9F")
ax.set_aspect("equal")
plt.show()

## 5. Top bairros e marcas

Estatísticas descritivas que vão ao `summary.json` (frontend pode
mostrar como "hot zones" ou em painéis laterais).

In [ ]:
print("Top 15 bairros:")
print(df_raw["BAIRRO"].str.strip().str.upper().value_counts().head(15))

print("\nTop 10 marcas:")
print(df_raw["MARCA_OBJETO"].value_counts().head(10))

---

**Próximo passo:** `02_limpeza.ipynb` — aplicar `cleaner.clean()` e salvar
parquet processado em `data/processed/`.